# E1 — Image classification with Softmax, MLP and CNN

Three models classify Fashion-MNIST from random initialization: a linear classifier over 784 pixels (Softmax), an MLP with dimensions 784→256→128→10, and a CNN with two convolutional blocks. The comparison examines how spatial structure affects predictive performance, parameter count and training time.

The full [experimental protocol](../docs/EXPERIMENTS.md) specifies the shared settings. AI assistance is disclosed in [AI_USE_REPORT.md](../AI_USE_REPORT.md).

## Experimental protocol

Each configuration is trained from random initialization for 30 epochs with batch size 256 and seeds 42, 43 and 44. The split and preprocessing are shared across configurations. Checkpoints are selected by validation accuracy, with lower validation cross-entropy breaking ties. The held-out test set is evaluated after the model choices and training settings are frozen.

In [ ]:
from pathlib import Path
import subprocess
import sys

colab_root = Path("/content/CO5085-deep-learning")
roots = (Path.cwd(), *Path.cwd().parents, colab_root)
ROOT = next(
    (path for path in roots if (path / "configs/experiments.json").is_file()), None
)
if ROOT is None and "google.colab" in sys.modules:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/dathuynh1108/CO5085-deep-learning.git",
            str(colab_root),
        ],
        check=True,
    )
    ROOT = colab_root
if ROOT is None:
    raise FileNotFoundError(
        "Open the notebook from the CO5085-deep-learning directory."
    )
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "notebooks"))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image
from IPython.display import display

from common.config import get_experiment
from common.config import load_experiments
from common.data import CLASS_NAMES
from common.data import prepare_data
from common.engine import train
from common.evaluation import evaluate_run
from common.factory import build_model
from common.factory import parameter_count
from common.runtime import atomic_csv
from common.runtime import atomic_json
from common.runtime import choose_device
from common.runtime import environment_info
from common.runtime import seed_everything
from exercise_notebook import flag
from exercise_notebook import read_json
from exercise_notebook import setting
from exercise_notebook import show_comparison
from exercise_notebook import show_figure
from exercise_notebook import show_results
from exercise_notebook import verified_training_status

EXERCISE = "E1"

In [ ]:
SEEDS = tuple(int(value) for value in setting("SEEDS", "42,43,44").split(","))
MODELS = setting("MODELS")
EPOCHS = int(setting("EPOCHS")) if setting("EPOCHS") else None
DEVICE = setting("DEVICE", "auto")
WORKERS = int(setting("WORKERS", "0"))
DATA_ROOT = (ROOT / setting("DATA_ROOT", "data")).resolve()
OUTPUT_ROOT = (ROOT / setting("OUTPUT_ROOT", "runs/notebooks/exercises")).resolve()
RESUME = flag("RESUME")
DETERMINISTIC = flag("DETERMINISTIC")
DOWNLOAD = flag("DOWNLOAD", default=True)
CONFIRM_TEST = flag("CONFIRM_TEST")

In [ ]:
if (
    not SEEDS
    or len(set(SEEDS)) != len(SEEDS)
    or any(not 0 <= seed < 2**32 for seed in SEEDS)
):
    raise ValueError("SEEDS must contain distinct seeds in [0, 2**32).")
SEEDS = tuple(sorted(SEEDS))
if EPOCHS is not None and OUTPUT_ROOT == (ROOT / "runs/notebooks/exercises").resolve():
    raise ValueError(
        "Changing EPOCHS requires a separate OUTPUT_ROOT for the check run."
    )
if OUTPUT_ROOT.is_relative_to((ROOT / "runs" / "exercises").resolve()):
    raise ValueError(
        "Choose an OUTPUT_ROOT outside the historical results directory runs/exercises."
    )
if WORKERS < 0:
    raise ValueError("WORKERS must be a non-negative integer.")
if CONFIRM_TEST and EPOCHS is not None:
    raise ValueError(
        "Runs with an EPOCHS override can only be evaluated on the validation set."
    )
device = choose_device(DEVICE)

In [ ]:
presets = load_experiments()
allowed = {row.id for row in presets if row.exercise == EXERCISE}
if EXERCISE == "E3":
    allowed.update({"e1_mlp", "e1_cnn"})
selected = (
    tuple(value.strip() for value in MODELS.split(","))
    if MODELS
    else tuple(row.id for row in presets if row.id in allowed)
)
if len(set(selected)) != len(selected) or set(selected) - allowed:
    raise ValueError(f"MODELS must contain distinct IDs from: {sorted(allowed)}")
configs = [get_experiment(identifier, epochs=EPOCHS) for identifier in selected]
display(
    {
        "device": str(device),
        "seed": SEEDS,
        "worker": WORKERS,
        "output_root": str(OUTPUT_ROOT),
    }
)
display(
    [
        {
            key: config.to_dict()[key]
            for key in (
                "id",
                "epochs",
                "batch_size",
                "lr",
                "tokenizer",
                "backend",
                "group",
            )
        }
        for config in configs
    ]
)

## Fashion-MNIST data exploration

Fashion-MNIST contains grayscale images of size 28×28 in ten clothing categories. A stratified split with seed 2026 divides the 60,000 official training images into 54,000 training and 6,000 validation images; each class contributes 5,400 and 600 images, respectively. The 10,000 test images remain held out. The plots below show class balance and examples from each training class.

Pixel values are scaled to [0,1] and standardized using the mean and standard deviation computed only from the training subset. The same normalization is applied to validation and test images. No data augmentation is used.

In [ ]:
split_seeds = {config.split_seed for config in configs}
if len(split_seeds) != 1:
    raise ValueError("Comparisons require the same data split.")
train_set, validation_set, dataset = prepare_data(
    DATA_ROOT, split_seeds.pop(), DOWNLOAD
)
train_counts = np.bincount(
    train_set.labels.numpy()[train_set.indices], minlength=len(CLASS_NAMES)
)
validation_counts = np.bincount(
    validation_set.labels.numpy()[validation_set.indices], minlength=len(CLASS_NAMES)
)
display(
    {
        key: dataset[key]
        for key in ("n_train", "n_validation", "mean", "std", "fingerprint")
    }
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
labels = np.arange(len(CLASS_NAMES))
ax.bar(labels - 0.18, train_counts, 0.36, label="Train")
ax.bar(labels + 0.18, validation_counts, 0.36, label="Validation")
ax.set_xticks(labels, CLASS_NAMES, rotation=30, ha="right")
ax.set(ylabel="Number of images", title="Class distribution by split")
ax.legend()
fig.tight_layout()
show_figure(fig)

In [ ]:
rng = np.random.default_rng(2026)
y = train_set.labels.numpy()
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for cls, ax in enumerate(axes.flat):
    index = int(rng.choice(train_set.indices[y[train_set.indices] == cls]))
    ax.imshow(train_set.images[index], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{CLASS_NAMES[cls]} · {index}", fontsize=9)
    ax.axis("off")
fig.tight_layout()
show_figure(fig)

## Model architectures

The Softmax classifier applies a linear map to the flattened image: $z=Wx+b$, with $x\in\mathbb{R}^{784}$. The MLP adds two hidden layers with ReLU and dropout. The CNN applies two convolutional blocks with batch normalization, ReLU and max pooling, followed by a classification head.

Models return logits. Cross-entropy computes $-z_y+\log\sum_k\exp(z_k)$, so no additional softmax is applied before the loss function.

In [ ]:
seed_everything(SEEDS[0], DETERMINISTIC)
images = torch.stack([train_set[i][0] for i in range(4)])
model_shapes = []
with torch.inference_mode():
    for config in configs:
        model = build_model(config).eval()
        row = {
            "experiment": config.id,
            "parameters": parameter_count(model),
            "logits": list(model(images).shape),
        }
        model_shapes.append(row)
display(model_shapes)

## Training and checkpoint selection

The [shared PyTorch loop](../src/common/engine.py) computes logits and cross-entropy, backpropagates the loss, clips the global gradient norm to 1, and updates parameters with AdamW. A cosine schedule reduces the learning rate to 1% of its initial value over 30 epochs. Validation accuracy and cross-entropy determine checkpoint selection; no early stopping is used.

In [ ]:
run_environment = environment_info(device)
directories = {}
for config in configs:
    for seed in SEEDS:
        directory = OUTPUT_ROOT / config.id / f"seed_{seed}"
        status = verified_training_status(
            directory,
            config,
            seed,
            dataset,
            WORKERS,
            DETERMINISTIC,
            run_environment,
        )
        if status == "trained":
            print(f"Reusing completed run: {config.id}, seed={seed}", flush=True)
        else:
            if status == "incomplete" and not RESUME:
                raise RuntimeError(
                    f"Incomplete run: {directory}. Enable RESUME to continue."
                )
            directory = train(
                config,
                seed,
                DATA_ROOT,
                OUTPUT_ROOT,
                device_name=DEVICE,
                workers=WORKERS,
                resume=status == "incomplete",
                deterministic=DETERMINISTIC,
                download=DOWNLOAD,
            )
        directories[(config.id, seed)] = directory

## Evaluation and error analysis

Accuracy, cross-entropy and macro-F1 summarize predictive performance. Parameter count and median training time per epoch describe model cost. Results are aggregated across seeds as a mean and sample standard deviation. Learning curves and confusion matrices use seed 42 in the full protocol. Confusion counts and up to eight of the most confident misclassifications characterize the remaining errors.

In [ ]:
split = "test" if CONFIRM_TEST else "validation"
report_dir = (
    OUTPUT_ROOT
    / "notebook_analysis"
    / EXERCISE
    / ("seeds_" + "_".join(map(str, SEEDS)))
    / split
)
evaluations, results = {}, []
for (identifier, seed), directory in directories.items():
    result = evaluate_run(
        directory,
        DATA_ROOT,
        split=split,
        confirm_test=CONFIRM_TEST,
        device_name=DEVICE,
        workers=WORKERS,
        download=DOWNLOAD,
    )
    evaluations[(identifier, seed)] = result
    training = read_json(directory / "metrics.json")
    results.append(
        {
            "experiment": identifier,
            "seed": seed,
            "split": split,
            "best_epoch": result["best_epoch"],
            "accuracy": result["accuracy"],
            "macro_f1": result["macro_f1"],
            "cross_entropy": result["cross_entropy"],
            "n_parameters": training["n_parameters"],
            "median_train_seconds": training["median_train_seconds"],
            "total_train_seconds": training["total_train_seconds"],
            "inference_images_per_second": result["inference_images_per_second"],
            "signature": result["signature"],
            "model_sha256": result["model_sha256"],
        }
    )

In [ ]:
atomic_csv(report_dir / "results.csv", results)
atomic_json(
    report_dir / "protocol.json",
    {
        "notebook": f"{EXERCISE}.ipynb",
        "configurations": [config.to_dict() for config in configs],
        "seeds": list(SEEDS),
        "dataset": dataset,
        "device": str(device),
        "workers": WORKERS,
        "deterministic": DETERMINISTIC,
        "split": split,
        "confirm_test": CONFIRM_TEST,
    },
)
show_results(results)

In [ ]:
representative_seed = min(SEEDS)
show_comparison(configs, directories, evaluations, representative_seed, report_dir)

In [ ]:
reference = configs[0].id
examples = (
    directories[(reference, representative_seed)] / "plots" / f"mistakes_{split}.png"
)
if examples.is_file():
    print(
        f"Errors with high prediction confidence: {reference}, seed={representative_seed}"
    )
    display(Image(filename=str(examples), width=650))

## Scope of the comparison

The linear classifier, MLP and CNN differ in both architecture and parameter count. Their results therefore describe the complete configurations rather than an isolated effect of convolution. The train–validation gap captures differences in fit, while confusion matrices distinguish recurring class ambiguities from aggregate accuracy.